# Urban Growth Potential Score Analysis

Exploratory review of the urban growth potential score.

Goals:
- Review the score distribution.
- Compare potential tiers against observed next-year urbanization.
- Inspect the highest-scoring cells overall and by city.
- Create quick maps for visual quality assurance.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(
        "Run this notebook from the repository root or from the notebooks directory."
    )

In [ ]:
potential_path = (
    PROJECT_ROOT
    / "data/predictions/potential/mx/500m"
    / "mx_urban_growth_potential_score_2016_2024_priorities_1_2_500m.parquet"
)
if not potential_path.exists():
    raise FileNotFoundError(f"Potential score file not found: {potential_path}")

df = gpd.read_parquet(potential_path)

required_columns = {
    "cell_id",
    "city_id",
    "year",
    "urban_growth_probability",
    "urban_growth_potential_score",
    "urban_growth_potential_percentile",
    "urban_growth_potential_tier",
    "urban_growth_potential_rank_city_year",
    "urbanized_next_year",
}
missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"Rows, columns: {df.shape}")
print(f"Cells: {df['cell_id'].nunique():,}")
print(f"Years: {sorted(df['year'].unique())}")
df.head()

## Overall Score Distribution

In [ ]:
df["urban_growth_potential_score"].describe()

In [ ]:
ax = df["urban_growth_potential_score"].hist(bins=50, figsize=(10, 5))
ax.set_title("Urban Growth Potential Score Distribution")
ax.set_xlabel("Score")
ax.set_ylabel("Cell-years")
plt.show()

## Performance by Tier

In [ ]:
tier_summary = (
    df.groupby("urban_growth_potential_tier")
    .agg(
        rows=("cell_id", "count"),
        positives=("urbanized_next_year", "sum"),
        positive_rate=("urbanized_next_year", "mean"),
        mean_score=("urban_growth_potential_score", "mean"),
        mean_probability=("urban_growth_probability", "mean"),
    )
    .sort_values("mean_score", ascending=False)
)

tier_summary

## Summary by Year

In [ ]:
year_summary = (
    df.groupby("year")
    .agg(
        rows=("cell_id", "count"),
        positives=("urbanized_next_year", "sum"),
        positive_rate=("urbanized_next_year", "mean"),
        mean_score=("urban_growth_potential_score", "mean"),
    )
    .reset_index()
)

year_summary

## Top 5% by Year

In [ ]:
top_5 = df[df["urban_growth_potential_percentile"] >= 0.95].copy()

top_5_summary = (
    top_5.groupby("year")
    .agg(
        rows=("cell_id", "count"),
        positives=("urbanized_next_year", "sum"),
        positive_rate=("urbanized_next_year", "mean"),
        mean_score=("urban_growth_potential_score", "mean"),
    )
    .reset_index()
)

top_5_summary

## Top Potential Cells in 2024

In [ ]:
cols = [
    "year",
    "city_id",
    "cell_id",
    "urban_growth_potential_score",
    "urban_growth_probability",
    "urban_growth_potential_tier",
    "built_probability_mean",
    "distance_to_nearest_road_m",
    "economic_business_count_total",
    "population_density_per_km2",
    "urbanized_next_year",
]

df[df["year"] == 2024].sort_values("urban_growth_potential_score", ascending=False)[cols].head(50)

## Top Potential Cells by City in 2024

In [ ]:
top_city_2024 = df[
    (df["year"] == 2024) & (df["urban_growth_potential_rank_city_year"] <= 10)
].sort_values(["city_id", "urban_growth_potential_rank_city_year"])

top_city_2024[cols + ["urban_growth_potential_rank_city_year"]]

## Quick City Map

Change `city_id` to inspect other cities.

In [ ]:
city_id = "mx_aguascalientes"
year = 2024

city = df[(df["city_id"] == city_id) & (df["year"] == year)].copy()
if city.empty:
    example_cities = sorted(df["city_id"].unique())[:10]
    raise ValueError(
        f"No rows found for city_id={city_id!r} and year={year}. Example city IDs: {example_cities}"
    )


ax = city.plot(
    column="urban_growth_potential_score",
    legend=True,
    figsize=(9, 9),
)
ax.set_title(f"Potential Score - {city_id} - {year}")
ax.set_axis_off()
plt.show()

## Very High Potential Cells in a City

In [ ]:
very_high_city = city[city["urban_growth_potential_tier"] == "very_high"].copy()

ax = city.boundary.plot(figsize=(9, 9), linewidth=0.2)
if very_high_city.empty:
    print(f"No very high potential cells found for {city_id} in {year}.")
else:
    very_high_city.plot(ax=ax, color="#d62728")
ax.set_title(f"Very High Potential Cells - {city_id} - {year}")
ax.set_axis_off()
plt.show()